Выбор большой языковой модели

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["NO_PROXY"] = (
    "localhost,127.0.0.1,::1,qdrant.example.com,192.168.1.100"
)

In [3]:
from llamba.chatmodels.ollama import OllamaModel

chatbot = OllamaModel(url="http://26.175.145.106:11434/",  #"http://127.0.0.1:11434/",
                      endpoint="api/generate",
                      model="qwen3:8b",  #"deepseek-r1",
                      num_threads=1,
                      check_connection_timeout=30,
                      request_timeout=300,
                      think=False)
connection = chatbot.check_connection()
print(connection)

True


Подготовка данных

In [4]:
import pandas as pd

mock_descriptions = {
    'CXCL9': {'description': 'gene for cancer', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'CCL22': {'description': 'gene for ccl', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'IL6': {'description': 'gene for fish', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'PDGFB': {'description': 'gene for idkB', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'CD40LG': {'description': 'gene for TV', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'IL27': {'description': 'gene for fishy fish', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'VEGFA': {'description': 'gene for monster', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'CSF1': {'description': 'gene for racing', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'PDGFA': {'description': 'gene for idkA', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'CXCL10': {'description': 'gene for 67', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'}}

desc_df = pd.DataFrame(mock_descriptions).T

In [5]:
desc_df

,description,low_shap_desc,high_shap_desc
CXCL9,gene for cancer,badbad,goodgood
CCL22,gene for ccl,badbad,goodgood
IL6,gene for fish,badbad,goodgood
PDGFB,gene for idkB,badbad,goodgood
CD40LG,gene for TV,badbad,goodgood
IL27,gene for fishy fish,badbad,goodgood
VEGFA,gene for monster,badbad,goodgood
CSF1,gene for racing,badbad,goodgood
PDGFA,gene for idkA,badbad,goodgood
CXCL10,gene for 67,badbad,goodgood


In [6]:
import torch
from torch import nn
import numpy as np

from llamba_library.bioage_model import BioAgeModel
from llamba_library.functions import get_shap_dict

#from txai_omics_3.models.tabular.widedeep.ft_transformer \
#    import WDFTTransformerModel, FN_SHAP, FN_CHECKPOINT, TRAIN_DATA_PATH


#### Данные
feats = [
    'CXCL9',
    'CCL22',
    'IL6',
    'PDGFB',
    'CD40LG',
    'IL27',
    'VEGFA',
    'CSF1',
    'PDGFA',
    'CXCL10'
]

#dataset_path = TRAIN_DATA_PATH
#train_data = pd.read_excel(TRAIN_DATA_PATH)
#train_data = train_data[train_data['Split'] == 'trn_val'].copy()
#train_data = train_data.loc[:, feats + ['Age']]

my_data = {'CXCL9': 2599.629474,
           'CCL22': 820.306524,
           'IL6': 0.846377,
           'PDGFB': 13400.666359,
           'CD40LG': 1853.847406,
           'IL27': 1128.886982,
           'VEGFA': 153.574220,
           'CSF1': 239.627236,
           'PDGFA': 1005.844290,
           'CXCL10': 228.229829,
           'Age': 90.454972}

data = pd.DataFrame(my_data, index=[0])


#### Модель
class DummyBioAgeModel(nn.Module):
    def forward(self, x):
        # Фиктивный возраст для проверки работы кода
        return torch.full(
            (x.shape[0], 1),
            50.0,
            dtype=x.dtype,
            device=x.device,
        )


model = DummyBioAgeModel()
bioage_model = BioAgeModel(model=model)


#fn_model = FN_CHECKPOINT
#model = WDFTTransformerModel.load_from_checkpoint(fn_model)
#bioage_model = BioAgeModel(model=model)

def predict_func(x):
    batch = {
        'all': torch.from_numpy(np.float32(x)),
        'continuous': torch.from_numpy(np.float32(x))
    }
    return model(batch).cpu().detach().numpy()


top_n = 3  # количество признаков с наибольшим вкладом

D:\job_isp\2026\app_hub\rag_app\src\llamba\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
data.T

,0
CXCL9,2599.629474
CCL22,820.306524
IL6,0.846377
PDGFB,13400.666359
CD40LG,1853.847406
IL27,1128.886982
VEGFA,153.574220
CSF1,239.627236
PDGFA,1005.844290
CXCL10,228.229829


In [8]:
desc_df

,description,low_shap_desc,high_shap_desc
CXCL9,gene for cancer,badbad,goodgood
CCL22,gene for ccl,badbad,goodgood
IL6,gene for fish,badbad,goodgood
PDGFB,gene for idkB,badbad,goodgood
CD40LG,gene for TV,badbad,goodgood
IL27,gene for fishy fish,badbad,goodgood
VEGFA,gene for monster,badbad,goodgood
CSF1,gene for racing,badbad,goodgood
PDGFA,gene for idkA,badbad,goodgood
CXCL10,gene for 67,badbad,goodgood


Инициализация коннектора

In [10]:
from llamba.connector import LlambaConnector

connector = LlambaConnector(language="en", bioage_model=bioage_model, chat_model=chatbot)

Инициализация векторной базы данных

In [11]:
from fastembed import TextEmbedding
from fastembed.common.model_description import (
    ModelSource,
    PoolingType,
)


TextEmbedding.add_custom_model(
    model="custom/bio_embedding_model",
    pooling=PoolingType.MEAN,
    normalization=True,
    sources=ModelSource(
        hf="pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
    ),
    dim=768,
    model_file="model.onnx",
)

In [31]:
from llamba.util.document_repo import DocumentRepoQdrant
from qdrant_client import QdrantClient
from fastembed import TextEmbedding, SparseTextEmbedding

dense_vectorizer = TextEmbedding(
    model_name='custom/bio_embedding_model',
    specific_model_path='../test_cfg/bio_embedding_model_onnx'
)
sparse_vectorizer = SparseTextEmbedding(
    model_name='Qdrant/bm25',
    specific_model_path='../test_cfg/bm25_en'
)

vectorizer_kwargs = {
    "dense_vectorizer": dense_vectorizer,
    "dense_field": "semantic",
    "sparse_embedder": sparse_vectorizer,
    "sparse_field": "sparse",

}


qdrant_client = QdrantClient(url="http://26.175.145.106", port=6333, api_key='test_api_key')
document_repo = DocumentRepoQdrant(client=qdrant_client,
                                   collection_name="command_created_collection",#"ageing_docs",
                                   payload_return="text",
                                   max_results=5,
                                   **vectorizer_kwargs
                                   )
connector.specify_db(document_repo)

C:\Users\VKN\AppData\Local\Temp\ipykernel_18392\1822693479.py:23: UserWarning: Api key is used with an insecure connection.
  qdrant_client = QdrantClient(url="http://26.175.145.106", port=6333, api_key='test_api_key')
D:\job_isp\2026\app_hub\rag_app\src\llamba\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:287: UserWarning: Qdrant client version 1.20.0 is incompatible with server version 1.18.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


Указание калькулятора биологических часов

In [26]:
connector.specify_clock_name(name="Immunoage", doi="10.1007/s11357-022-00540-4")

Указание производственных условий

In [27]:
connector.specify_conditions(condition="mining industry", duration="15 years")

D:\job_isp\2026\app_hub\rag_app\src\llamba\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:287: UserWarning: Qdrant client version 1.20.0 is incompatible with server version 1.18.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


Передача экспертной системе данных 

In [28]:
res = connector.analyze(data=data, top_n=3,  #analyze_feats=True, analyze_risks=True, train_data=train_data,
                        predict_func=predict_func)
print(res['analysis'])

1) Bioage: 50. Age acceleration: -40. This indicates that the individual is aging slower than their chronological age.  

2)  
| Feature | Value | Age acceleration |  
|--------|-------|------------------|  
| Bioage | 50 | -40 |  
| Physical activity | High | -40 |  
| Diet quality | Balanced | -40 |  
| Sleep quality | Good | -40 |  
| Stress levels | Low | -40 |  

3)  
- **Bioage**: 50. A lower bioage compared to chronological age indicates slower aging.  
- **Physical activity**: High. Regular physical activity is associated with slower aging and reduced disease risk.  
- **Diet quality**: Balanced. A balanced diet supports cellular health and reduces aging acceleration.  
- **Sleep quality**: Good. Poor sleep accelerates aging, while good sleep supports cellular repair.  
- **Stress levels**: Low. Chronic stress increases aging acceleration and disease risk.  

4)  
| Disease | Risk | Comments |  
|--------|------|----------|  
| Cardiovascular disease | Low | Risk is reduced due

In [33]:
connector.document_repo.search("""You are given a data report. Make sure that it is in en. Structure it in the following way: 1) Show bioage and age acceleration.2) List the most influential parameters and their values as a table with the Feature, Value, Age acceleration columns.3) Give a short description for each most influential parameter and what it means if the value is greater/smaller than normal.4) Provide information about the risk of disease occurrence as a table with the Disease, Risk, Comments columns.5) Describe in detail how each disease is associated with parameters.The report is as follows: Your bioage is 50 and your aging acceleration is -40, which means you are ageing slower than normal.

Your bioage is 50 and your aging acceleration is -40, which means you are ageing slower than normal.

Your bioage is 50 and your aging acceleration is -40, which means you are ageing slower than normal.

""")

['According to the general meaning of the attribute, it is possible to understand in which range its values lie. When receiving data from a source, according to these criteria, it is possible to determine the correctness of the data provided or to find anomalies in the data. As with any set of numerical values, various functions can be applied to numerical ones, finding important statistical parameters of the data provided. Thus, it is possible to estimate the mean value, variance and other possible parameters of distributions that can give a general idea of the feature under consideration.',
 'The statistical language of the BayesExpert dependency rule is that of the medical literature, including sensitivity and specificity, relative risk, and their confidence intervals. These rules express dependent relations as one would expect in a Bayesian net, translated from these statistics. In addition, the scientist can use "and" and "or" from standard logic as well as other functions like "a

In [30]:
recommendations = connector.produce_recommendations()
print(recommendations)

1. **Enhance Nutritional Intake**: Given the individual's history in the mining industry, exposure to heavy metals and environmental toxins may have contributed to oxidative stress and inflammation. A diet rich in antioxidants (e.g., berries, leafy greens, nuts) and anti-inflammatory foods (e.g., fatty fish, turmeric, ginger) can mitigate cellular damage and support immune function. Nutrient-dense foods such as whole grains, legumes, and lean proteins should be prioritized to ensure adequate intake of vitamins (especially B-complex, C, E) and minerals (e.g., selenium, zinc) that support immune health and reduce inflammation.

2. **Regular Physical Activity**: Engaging in moderate-intensity aerobic exercise (e.g., brisk walking, cycling, swimming) for at least 150 minutes per week can improve cardiovascular health, enhance mitochondrial function, and reduce systemic inflammation. Resistance training (e.g., weightlifting, bodyweight exercises) should also be incorporated to maintain musc